# TS2Vec + XGBoost Ensemble for Better Forecasting Results

This notebook implements an ensemble approach that combines:
- **TS2Vec baseline model** (deep learning representations)
- **XGBoost regressor** (gradient boosting)
- **Ridge regression** (linear baseline)
- **Weighted ensemble averaging** for optimal combination

**Goal**: Achieve better test set performance than TS2Vec baseline alone by leveraging ensemble diversity.

**Dataset**: ETTh2 univariate forecasting with horizons H=[24, 48, 168, 336, 720]

In [ ]:
# Complete TS2Vec Implementation with Ensemble
# Self-contained implementation - no external dependencies on TS2Vec repository

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import time
import os
import math
import random
from datetime import datetime
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW

# Machine Learning for Ensemble
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV

# XGBoost for ensemble
try:
    import xgboost as xgb
    HAS_XGB = True
    print("✅ XGBoost available")
except ImportError:
    HAS_XGB = False
    print("⚠️ XGBoost not found, using GradientBoostingRegressor")

# Set device and random seeds
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print(f"✅ All libraries imported successfully!")
print(f"? Using device: {device}")
print(f"🧠 PyTorch version: {torch.__version__}")
print("🚀 Ready to implement TS2Vec + Ensemble from scratch!")

In [ ]:
# Configuration
DATASET = 'ETTh2'
UNIVAR = True  # Univariate forecasting
QUICK_MODE = True  # Set False for full training

# TS2Vec Model Parameters
TS2VEC_EPOCHS = 100 if QUICK_MODE else 200
REPR_DIMS = 320
HIDDEN_DIMS = 64
ENCODER_DEPTH = 10
BATCH_SIZE = 8
MAX_TRAIN_LENGTH = None
LEARNING_RATE = 0.001
TEMPORAL_UNIT = 0

# Ensemble Parameters
ENSEMBLE_WEIGHTS = {'ts2vec': 0.4, 'xgb': 0.4, 'ridge': 0.2}
N_ESTIMATORS = 200 if QUICK_MODE else 500

# Forecasting Parameters  
INPUT_LENGTH = 168  # Past 7 days
PREDICTION_HORIZONS = [24, 48, 168, 336, 720]  # ETTh prediction horizons
PADDING = 200  # For sliding window encoding

print(f"📊 Dataset: {DATASET}")
print(f"🔬 Univariate: {UNIVAR}")  
print(f"⚡ Quick mode: {QUICK_MODE}")
print(f"🤖 TS2Vec epochs: {TS2VEC_EPOCHS}")
print(f"? Repr dims: {REPR_DIMS}, Hidden dims: {HIDDEN_DIMS}")
print(f"🏗️ Encoder depth: {ENCODER_DEPTH}")
print(f"🎯 Prediction horizons: {PREDICTION_HORIZONS}")
print(f"🔥 Device: {device}")

In [ ]:
# Data Loading and Preprocessing Functions
def load_ETTh_data(dataset_name='ETTh2', univar=True):
    """Load ETTh dataset with proper preprocessing"""
    
    # Try to load from different possible paths
    possible_paths = [
        f'./datasets/{dataset_name}.csv',
        f'./{dataset_name}.csv', 
        f'datasets/{dataset_name}.csv',
        f'{dataset_name}.csv'
    ]
    
    data_file = None
    for path in possible_paths:
        if os.path.exists(path):
            data_file = path
            break
    
    if data_file is None:
        print(f"❌ {dataset_name}.csv not found in any of these locations:")
        for path in possible_paths:
            print(f"   - {path}")
        raise FileNotFoundError(f"Please download {dataset_name}.csv and place it in the datasets/ folder")
    
    print(f"✅ Loading data from: {data_file}")
    
    # Load the CSV file
    data = pd.read_csv(data_file, index_col='date', parse_dates=True)
    
    # Generate time features
    dt_embed = np.stack([
        data.index.minute.to_numpy(),
        data.index.hour.to_numpy(),
        data.index.dayofweek.to_numpy(),
        data.index.day.to_numpy(),
        data.index.dayofyear.to_numpy(),
        data.index.month.to_numpy(),
        data.index.isocalendar().week.to_numpy(),
    ], axis=1).astype(np.float32)
    
    n_covariate_cols = dt_embed.shape[-1]
    
    # Select univariate or multivariate
    if univar:
        if dataset_name in ('ETTh1', 'ETTh2', 'ETTm1', 'ETTm2'):
            data = data[['OT']]  # Oil Temperature
        else:
            data = data.iloc[:, -1:]  # Last column
    
    # Convert to numpy
    data_values = data.to_numpy().astype(np.float32)
    
    # Define train/valid/test splits for ETT datasets
    if dataset_name in ('ETTh1', 'ETTh2'):
        train_slice = slice(None, 12*30*24)      # 12 months training
        valid_slice = slice(12*30*24, 16*30*24)  # 4 months validation  
        test_slice = slice(16*30*24, 20*30*24)   # 4 months test
    else:
        # Generic split
        train_slice = slice(None, int(0.6 * len(data_values)))
        valid_slice = slice(int(0.6 * len(data_values)), int(0.8 * len(data_values)))
        test_slice = slice(int(0.8 * len(data_values)), None)
    
    # Fit scaler on training data only
    scaler = StandardScaler().fit(data_values[train_slice])
    data_scaled = scaler.transform(data_values)
    
    # Add batch dimension and combine with time features
    data_scaled = np.expand_dims(data_scaled, 0)  # Shape: (1, T, D)
    
    if n_covariate_cols > 0:
        dt_scaler = StandardScaler().fit(dt_embed[train_slice])
        dt_embed_scaled = np.expand_dims(dt_scaler.transform(dt_embed), 0)
        data_final = np.concatenate([dt_embed_scaled, data_scaled], axis=-1)
    else:
        data_final = data_scaled
        
    return data_final, train_slice, valid_slice, test_slice, scaler, PREDICTION_HORIZONS, n_covariate_cols

print("🔧 Data loading functions defined!")

In [ ]:
# TS2Vec Architecture Implementation

class SamePadConv(nn.Module):
    """1D Convolution with same padding"""
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1, groups=1):
        super().__init__()
        self.receptive_field = (kernel_size - 1) * dilation + 1
        padding = self.receptive_field // 2
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation, groups=groups
        )
        self.remove = 1 if self.receptive_field % 2 == 0 else 0
        
    def forward(self, x):
        out = self.conv(x)
        if self.remove > 0:
            out = out[:, :, :-self.remove]
        return out

class ConvBlock(nn.Module):
    """Dilated convolution block with residual connection"""
    def __init__(self, in_channels, out_channels, kernel_size, dilation, final=False):
        super().__init__()
        self.conv1 = SamePadConv(in_channels, out_channels, kernel_size, dilation=dilation)
        self.conv2 = SamePadConv(out_channels, out_channels, kernel_size, dilation=dilation)
        self.projector = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels or final else None
    
    def forward(self, x):
        residual = x if self.projector is None else self.projector(x)
        x = F.gelu(x)
        x = self.conv1(x)
        x = F.gelu(x)
        x = self.conv2(x)
        return x + residual

class DilatedConvEncoder(nn.Module):
    """Dilated convolution encoder"""
    def __init__(self, in_channels, channels, kernel_size=3):
        super().__init__()
        self.net = nn.Sequential(*[
            ConvBlock(
                channels[i-1] if i > 0 else in_channels,
                channels[i],
                kernel_size=kernel_size,
                dilation=2**i,
                final=(i == len(channels)-1)
            )
            for i in range(len(channels))
        ])
        
    def forward(self, x):
        return self.net(x)

print("🏗️ TS2Vec architecture components defined!")
print("   - SamePadConv: 1D convolution with same padding")  
print("   - ConvBlock: Dilated convolution with residual connections")
print("   - DilatedConvEncoder: Multi-scale feature extraction")

In [ ]:
# TS2Vec Encoder and Loss Functions

def generate_binomial_mask(B, T, p=0.5):
    """Generate random binomial mask for training"""
    return torch.from_numpy(np.random.binomial(1, p, size=(B, T))).to(torch.bool)

class TSEncoder(nn.Module):
    """TS2Vec Time Series Encoder"""
    def __init__(self, input_dims, output_dims, hidden_dims=64, depth=10, mask_mode='binomial'):
        super().__init__()
        self.input_dims = input_dims
        self.output_dims = output_dims
        self.hidden_dims = hidden_dims
        self.mask_mode = mask_mode
        
        self.input_fc = nn.Linear(input_dims, hidden_dims)
        self.feature_extractor = DilatedConvEncoder(
            hidden_dims,
            [hidden_dims] * depth + [output_dims],
            kernel_size=3
        )
        self.repr_dropout = nn.Dropout(p=0.1)
        
    def forward(self, x, mask=None):  
        # x: B x T x input_dims
        nan_mask = ~x.isnan().any(axis=-1)
        x[~nan_mask] = 0
        x = self.input_fc(x)  # B x T x hidden_dims
        
        # Generate mask for training
        if mask is None:
            if self.training:
                mask = self.mask_mode
            else:
                mask = 'all_true'
        
        if mask == 'binomial':
            mask = generate_binomial_mask(x.size(0), x.size(1)).to(x.device)
        elif mask == 'all_true':
            mask = x.new_full((x.size(0), x.size(1)), True, dtype=torch.bool)
        elif mask == 'all_false':
            mask = x.new_full((x.size(0), x.size(1)), False, dtype=torch.bool)
        
        mask &= nan_mask
        x[~mask] = 0
        
        # Apply dilated convolutions
        x = x.transpose(1, 2)  # B x hidden_dims x T
        x = self.repr_dropout(self.feature_extractor(x))  # B x output_dims x T
        x = x.transpose(1, 2)  # B x T x output_dims
        
        return x

print("🧠 TSEncoder defined!")
print("   - Input projection to hidden dimensions")
print("   - Dilated convolution feature extraction")
print("   - Masking support for contrastive learning")
print("   - Dropout regularization")

In [ ]:
# Contrastive Loss Functions for TS2Vec

def instance_contrastive_loss(z1, z2):
    """Instance-level contrastive loss"""
    B, T = z1.size(0), z1.size(1)
    if B == 1:
        return z1.new_tensor(0.)
    
    z = torch.cat([z1, z2], dim=0)  # 2B x T x C
    z = z.transpose(0, 1)  # T x 2B x C
    sim = torch.matmul(z, z.transpose(1, 2))  # T x 2B x 2B
    
    logits = torch.tril(sim, diagonal=-1)[:, :, :-1]    # T x 2B x (2B-1)
    logits += torch.triu(sim, diagonal=1)[:, :, 1:]
    logits = -F.log_softmax(logits, dim=-1)
    
    i = torch.arange(B, device=z1.device)
    loss = (logits[:, i, B + i - 1].mean() + logits[:, B + i, i].mean()) / 2
    return loss

def temporal_contrastive_loss(z1, z2):
    """Temporal-level contrastive loss"""
    B, T = z1.size(0), z1.size(1)
    if T == 1:
        return z1.new_tensor(0.)
    
    z = torch.cat([z1, z2], dim=1)  # B x 2T x C
    sim = torch.matmul(z, z.transpose(1, 2))  # B x 2T x 2T
    
    logits = torch.tril(sim, diagonal=-1)[:, :, :-1]    # B x 2T x (2T-1)
    logits += torch.triu(sim, diagonal=1)[:, :, 1:]
    logits = -F.log_softmax(logits, dim=-1)
    
    t = torch.arange(T, device=z1.device)
    loss = (logits[:, t, T + t - 1].mean() + logits[:, T + t, t].mean()) / 2
    return loss

def hierarchical_contrastive_loss(z1, z2, alpha=0.5, temporal_unit=0):
    """Hierarchical contrastive loss combining instance and temporal levels"""
    loss = torch.tensor(0., device=z1.device)
    d = 0
    
    while z1.size(1) > 1:
        if alpha != 0:
            loss += alpha * instance_contrastive_loss(z1, z2)
        if d >= temporal_unit:
            if 1 - alpha != 0:
                loss += (1 - alpha) * temporal_contrastive_loss(z1, z2)
        d += 1
        z1 = F.max_pool1d(z1.transpose(1, 2), kernel_size=2).transpose(1, 2)
        z2 = F.max_pool1d(z2.transpose(1, 2), kernel_size=2).transpose(1, 2)
    
    if z1.size(1) == 1:
        if alpha != 0:
            loss += alpha * instance_contrastive_loss(z1, z2)
        d += 1
        
    return loss / d

print("🎯 Contrastive loss functions defined!")
print("   - Instance contrastive loss: learns to distinguish different samples")
print("   - Temporal contrastive loss: learns temporal relationships")  
print("   - Hierarchical loss: combines multi-scale contrastive learning")

In [ ]:
# Complete TS2Vec Model Implementation

def take_per_row(A, indx, num_elem):
    """Take num_elem elements from each row of A starting at positions in indx"""
    all_indx = indx[:, None] + np.arange(num_elem)
    return A[torch.arange(all_indx.shape[0])[:, None], all_indx]

def torch_pad_nan(x, left=0, right=0, dim=0):
    """Pad tensor with NaN values"""
    if left == 0 and right == 0:
        return x
    pad_shape = list(x.shape)
    pad_shape[dim] = left + right
    pad = torch.full(pad_shape, float('nan'), dtype=x.dtype, device=x.device)
    
    if left > 0 and right > 0:
        return torch.cat([pad[:, :left], x, pad[:, -right:]], dim=dim)
    elif left > 0:
        return torch.cat([pad[:, :left], x], dim=dim)
    elif right > 0:
        return torch.cat([x, pad[:, -right:]], dim=dim)

class TS2Vec:
    """Complete TS2Vec model for time series representation learning"""
    
    def __init__(self, input_dims, output_dims=320, hidden_dims=64, depth=10, 
                 device='cuda', lr=0.001, batch_size=16, max_train_length=None, temporal_unit=0):
        self.device = device
        self.lr = lr
        self.batch_size = batch_size
        self.max_train_length = max_train_length
        self.temporal_unit = temporal_unit
        
        # Create encoder
        self._net = TSEncoder(
            input_dims=input_dims, 
            output_dims=output_dims, 
            hidden_dims=hidden_dims, 
            depth=depth
        ).to(self.device)
        
        # Use Stochastic Weight Averaging for better generalization
        self.net = torch.optim.swa_utils.AveragedModel(self._net)
        self.net.update_parameters(self._net)
        
        self.n_epochs = 0
        self.n_iters = 0
        
    def fit(self, train_data, n_epochs=None, n_iters=None, verbose=False):
        """Train the TS2Vec model"""
        assert train_data.ndim == 3
        
        if n_iters is None and n_epochs is None:
            n_iters = 200 if train_data.size <= 100000 else 600
            
        # Handle long sequences by splitting
        if self.max_train_length is not None:
            sections = train_data.shape[1] // self.max_train_length
            if sections >= 2:
                train_data_split = []
                for i in range(sections):
                    start_idx = i * self.max_train_length
                    end_idx = (i + 1) * self.max_train_length
                    train_data_split.append(train_data[:, start_idx:end_idx])
                train_data = np.concatenate(train_data_split, axis=0)
        
        # Remove completely missing samples
        train_data = train_data[~np.isnan(train_data).all(axis=2).all(axis=1)]
        
        # Create data loader
        train_dataset = TensorDataset(torch.from_numpy(train_data).to(torch.float))
        train_loader = DataLoader(train_dataset, batch_size=min(self.batch_size, len(train_dataset)), 
                                shuffle=True, drop_last=True)
        
        # Initialize optimizer
        optimizer = AdamW(self._net.parameters(), lr=self.lr)
        
        loss_log = []
        
        print(f"🚀 Starting TS2Vec training...")
        print(f"   Data shape: {train_data.shape}")
        print(f"   Batch size: {min(self.batch_size, len(train_dataset))}")
        print(f"   Device: {self.device}")
        
        while True:
            if n_epochs is not None and self.n_epochs >= n_epochs:
                break
                
            cum_loss = 0
            n_epoch_iters = 0
            interrupted = False
            
            for batch in train_loader:
                if n_iters is not None and self.n_iters >= n_iters:
                    interrupted = True
                    break
                    
                x = batch[0]
                if self.max_train_length is not None and x.size(1) > self.max_train_length:
                    window_offset = np.random.randint(x.size(1) - self.max_train_length + 1)
                    x = x[:, window_offset:window_offset + self.max_train_length]
                
                x = x.to(self.device)
                
                # Generate random crops for contrastive learning
                ts_l = x.size(1)
                crop_l = np.random.randint(low=2 ** (self.temporal_unit + 1), high=ts_l + 1)
                crop_left = np.random.randint(ts_l - crop_l + 1)
                crop_right = crop_left + crop_l
                crop_eleft = np.random.randint(crop_left + 1)
                crop_eright = np.random.randint(low=crop_right, high=ts_l + 1)
                crop_offset = np.random.randint(low=-crop_eleft, high=ts_l - crop_eright + 1, size=x.size(0))
                
                optimizer.zero_grad()
                
                # Forward pass for two augmented views
                out1 = self._net(take_per_row(x, crop_offset + crop_eleft, crop_right - crop_eleft))
                out1 = out1[:, -crop_l:]
                
                out2 = self._net(take_per_row(x, crop_offset + crop_left, crop_eright - crop_left))
                out2 = out2[:, :crop_l]
                
                # Compute contrastive loss
                loss = hierarchical_contrastive_loss(out1, out2, temporal_unit=self.temporal_unit)
                
                # Backward pass
                loss.backward()
                optimizer.step()
                self.net.update_parameters(self._net)
                
                cum_loss += loss.item()
                n_epoch_iters += 1
                self.n_iters += 1
            
            if interrupted:
                break
                
            cum_loss /= n_epoch_iters
            loss_log.append(cum_loss)
            
            if verbose:
                print(f"Epoch #{self.n_epochs}: loss={cum_loss:.6f}")
                
            self.n_epochs += 1
        
        print(f"✅ Training completed! Final loss: {loss_log[-1]:.6f}")
        return loss_log

print("🤖 Complete TS2Vec model implemented!")
print("   - Contrastive self-supervised learning")
print("   - Hierarchical temporal modeling")
print("   - Stochastic Weight Averaging for stability")

In [ ]:
# TS2Vec Encoding and Forecasting Functions

def encode_ts2vec(ts2vec_model, data, batch_size=None, sliding_length=1, sliding_padding=200):
    """Encode time series using trained TS2Vec model"""
    if batch_size is None:
        batch_size = ts2vec_model.batch_size
    
    n_samples, ts_l, _ = data.shape
    
    # Set to evaluation mode
    ts2vec_model.net.eval()
    
    dataset = TensorDataset(torch.from_numpy(data).to(torch.float))
    loader = DataLoader(dataset, batch_size=batch_size)
    
    with torch.no_grad():
        output = []
        for batch in loader:
            x = batch[0]
            
            # Sliding window encoding for causal representation
            if sliding_length is not None:
                reprs = []
                for i in range(0, ts_l, sliding_length):
                    l = i - sliding_padding
                    r = i + sliding_length
                    x_sliding = torch_pad_nan(
                        x[:, max(l, 0):min(r, ts_l)],
                        left=-l if l < 0 else 0,
                        right=r - ts_l if r > ts_l else 0,
                        dim=1
                    )
                    
                    out = ts2vec_model.net(x_sliding.to(ts2vec_model.device))
                    out = out[:, sliding_padding:sliding_padding + sliding_length]
                    reprs.append(out.cpu())
                
                out = torch.cat(reprs, dim=1)
            else:
                out = ts2vec_model.net(x.to(ts2vec_model.device))
                out = out.cpu()
            
            output.append(out)
    
    output = torch.cat(output, dim=0)
    return output.numpy()

def generate_pred_samples(features, data, pred_len, drop=0):
    """Generate supervised learning samples for forecasting"""
    n = data.shape[1]
    features = features[:, :-pred_len]
    labels = np.stack([data[:, i:1 + n + i - pred_len] for i in range(pred_len)], axis=2)[:, 1:]
    features = features[:, drop:]
    labels = labels[:, drop:]
    return features.reshape(-1, features.shape[-1]), labels.reshape(-1, labels.shape[2] * labels.shape[3])

def cal_metrics(pred, target):
    """Calculate forecasting metrics"""
    return {
        'MSE': ((pred - target) ** 2).mean(),
        'MAE': np.abs(pred - target).mean()
    }

def fit_ridge_regressor(train_X, train_y, valid_X, valid_y):
    """Fit Ridge regression with validation-based alpha selection"""
    alphas = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
    best_alpha = 1.0
    best_score = float('inf')
    
    for alpha in alphas:
        ridge = Ridge(alpha=alpha)
        ridge.fit(train_X, train_y)
        valid_pred = ridge.predict(valid_X)
        score = mean_squared_error(valid_y, valid_pred)
        if score < best_score:
            best_score = score
            best_alpha = alpha
    
    # Fit with best alpha on combined data
    ridge = Ridge(alpha=best_alpha)
    X_combined = np.vstack([train_X, valid_X])
    y_combined = np.vstack([train_y, valid_y])
    ridge.fit(X_combined, y_combined)
    
    return ridge

print("🔍 TS2Vec encoding and forecasting functions defined!")
print("   - Sliding window causal encoding")
print("   - Supervised sample generation")
print("   - Ridge regression with validation")
print("   - Metric calculation (MSE, MAE)")

In [ ]:
# Complete Ensemble Implementation

class TS2VecEnsemble:
    """Ensemble combining TS2Vec with XGBoost and other ML models"""
    
    def __init__(self, ensemble_weights=None, use_xgb=True):
        self.ensemble_weights = ensemble_weights or ENSEMBLE_WEIGHTS
        self.use_xgb = use_xgb and HAS_XGB
        self.models = {}
        self.scalers = {}
        
    def create_models(self):
        """Create ensemble models"""
        models = {}
        
        # XGBoost or Gradient Boosting
        if self.use_xgb:
            models['xgb'] = MultiOutputRegressor(
                xgb.XGBRegressor(
                    n_estimators=N_ESTIMATORS,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    random_state=42,
                    tree_method='hist',
                    n_jobs=-1
                )
            )
        else:
            models['xgb'] = MultiOutputRegressor(
                GradientBoostingRegressor(
                    n_estimators=N_ESTIMATORS // 2,
                    max_depth=6,
                    learning_rate=0.1,
                    random_state=42
                )
            )
            
        # Random Forest
        models['rf'] = MultiOutputRegressor(
            RandomForestRegressor(
                n_estimators=N_ESTIMATORS // 2,
                max_depth=8,
                min_samples_split=5,
                random_state=42,
                n_jobs=-1
            )
        )
        
        return models
    
    def fit_ensemble(self, train_X, train_y, valid_X, valid_y):
        """Fit all ensemble models"""
        print("   🔧 Training Ridge regression (TS2Vec baseline)...")
        self.models['ridge'] = fit_ridge_regressor(train_X, train_y, valid_X, valid_y)
        
        # Combine train and validation for tree-based models
        X_combined = np.vstack([train_X, valid_X])
        y_combined = np.vstack([train_y, valid_y])
        
        # Scale features for XGBoost
        self.scalers['xgb'] = StandardScaler()
        X_scaled = self.scalers['xgb'].fit_transform(X_combined)
        
        print("   🔧 Training XGBoost...")
        ml_models = self.create_models()
        self.models['xgb'] = ml_models['xgb']
        self.models['xgb'].fit(X_scaled, y_combined)
        
        print("   🔧 Training Random Forest...")
        self.models['rf'] = ml_models['rf']
        self.models['rf'].fit(X_combined, y_combined)  # RF handles unscaled features well
        
        return self
    
    def predict_ensemble(self, test_X):
        """Generate ensemble predictions"""
        predictions = {}
        
        # TS2Vec + Ridge prediction
        predictions['ridge'] = self.models['ridge'].predict(test_X)
        
        # XGBoost prediction (with scaling)
        test_X_scaled = self.scalers['xgb'].transform(test_X)
        predictions['xgb'] = self.models['xgb'].predict(test_X_scaled)
        
        # Random Forest prediction
        predictions['rf'] = self.models['rf'].predict(test_X)
        
        # Weighted ensemble
        weights = self.ensemble_weights
        ensemble_pred = (
            weights.get('ridge', 0.3) * predictions['ridge'] +
            weights.get('xgb', 0.4) * predictions['xgb'] +
            weights.get('rf', 0.3) * predictions['rf']
        )
        
        return ensemble_pred, predictions

# Main Execution Function
def run_ts2vec_ensemble_forecasting():
    """Complete workflow: Load data -> Train TS2Vec -> Ensemble -> Evaluate"""
    
    print("="*80)
    print("🚀 STARTING TS2VEC + ENSEMBLE FORECASTING")
    print("="*80)
    
    # 1. Load Data
    print(f"\n📥 Loading {DATASET} dataset...")
    try:
        data, train_slice, valid_slice, test_slice, scaler, pred_lens, n_covariate_cols = \
            load_ETTh_data(DATASET, UNIVAR)
        
        print(f"✅ Data loaded successfully!")
        print(f"   Shape: {data.shape}")
        print(f"   Train: {train_slice}")  
        print(f"   Valid: {valid_slice}")
        print(f"   Test: {test_slice}")
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None
    
    # 2. Train TS2Vec Model
    print(f"\n🤖 Training TS2Vec model...")
    ts2vec_model = TS2Vec(
        input_dims=data.shape[-1],
        output_dims=REPR_DIMS,
        hidden_dims=HIDDEN_DIMS,
        depth=ENCODER_DEPTH,
        device=device,
        lr=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        max_train_length=MAX_TRAIN_LENGTH,
        temporal_unit=TEMPORAL_UNIT
    )
    
    train_data = data[:, train_slice]
    start_time = time.time()
    loss_log = ts2vec_model.fit(train_data, n_epochs=TS2VEC_EPOCHS, verbose=True)
    training_time = time.time() - start_time
    
    print(f"✅ TS2Vec training completed in {training_time:.2f}s")
    
    # 3. Extract Representations
    print(f"\n🔍 Extracting TS2Vec representations...")
    start_time = time.time()
    all_repr = encode_ts2vec(ts2vec_model, data, sliding_length=1, sliding_padding=PADDING)
    encoding_time = time.time() - start_time
    
    print(f"✅ Representations extracted in {encoding_time:.2f}s")
    print(f"   Representation shape: {all_repr.shape}")
    
    return {
        'data': data,
        'train_slice': train_slice,
        'valid_slice': valid_slice, 
        'test_slice': test_slice,
        'scaler': scaler,
        'pred_lens': pred_lens,
        'n_covariate_cols': n_covariate_cols,
        'ts2vec_model': ts2vec_model,
        'all_repr': all_repr,
        'loss_log': loss_log,
        'training_time': training_time,
        'encoding_time': encoding_time
    }

print("🏗️ Complete TS2Vec + Ensemble system implemented!")
print("   - TS2Vec representation learning")
print("   - Ridge + XGBoost + Random Forest ensemble")
print("   - Weighted prediction combination")
print("   - Ready for forecasting evaluation!")

In [ ]:
# Execute Complete TS2Vec + Ensemble Workflow and Evaluation

def evaluate_forecasting(results_dict):
    """Evaluate forecasting performance on multiple horizons"""
    
    # Extract results
    data = results_dict['data']
    train_slice = results_dict['train_slice'] 
    valid_slice = results_dict['valid_slice']
    test_slice = results_dict['test_slice']
    scaler = results_dict['scaler']
    pred_lens = results_dict['pred_lens']
    n_covariate_cols = results_dict['n_covariate_cols']
    all_repr = results_dict['all_repr']
    
    # Split representations and data
    train_repr = all_repr[:, train_slice]
    valid_repr = all_repr[:, valid_slice]
    test_repr = all_repr[:, test_slice]
    
    train_data = data[:, train_slice, n_covariate_cols:]
    valid_data = data[:, valid_slice, n_covariate_cols:]
    test_data = data[:, test_slice, n_covariate_cols:]
    
    # Evaluation on multiple horizons
    eval_horizons = [24, 48, 168] if QUICK_MODE else pred_lens
    print(f"\n🎯 Evaluating on horizons: {eval_horizons}")
    
    baseline_results = {}
    ensemble_results = {}
    individual_results = {}
    
    for pred_len in eval_horizons:
        print(f"\n{'='*50}")
        print(f"📊 Horizon H={pred_len}")
        print(f"{'='*50}")
        
        # Generate supervised samples
        train_X, train_y = generate_pred_samples(train_repr, train_data, pred_len, drop=PADDING)
        valid_X, valid_y = generate_pred_samples(valid_repr, valid_data, pred_len)
        test_X, test_y = generate_pred_samples(test_repr, test_data, pred_len)
        
        print(f"   Train: {train_X.shape}, Valid: {valid_X.shape}, Test: {test_X.shape}")
        
        # Baseline: TS2Vec + Ridge only
        print(f"   🔍 Training baseline (TS2Vec + Ridge)...")
        baseline_model = fit_ridge_regressor(train_X, train_y, valid_X, valid_y)
        baseline_pred = baseline_model.predict(test_X)
        
        # Ensemble: TS2Vec + Ridge + XGBoost + RF
        print(f"   🚀 Training ensemble models...")
        ensemble = TS2VecEnsemble(ensemble_weights=ENSEMBLE_WEIGHTS)
        ensemble.fit_ensemble(train_X, train_y, valid_X, valid_y)
        ensemble_pred, individual_preds = ensemble.predict_ensemble(test_X)
        
        # Reshape for evaluation
        ori_shape = (test_data.shape[0], -1, pred_len, test_data.shape[2])
        baseline_reshaped = baseline_pred.reshape(ori_shape)
        ensemble_reshaped = ensemble_pred.reshape(ori_shape)
        test_y_reshaped = test_y.reshape(ori_shape)
        
        # Calculate metrics
        baseline_results[pred_len] = {
            'norm': cal_metrics(baseline_reshaped, test_y_reshaped)
        }
        
        ensemble_results[pred_len] = {
            'norm': cal_metrics(ensemble_reshaped, test_y_reshaped)
        }
        
        # Individual model results
        individual_results[pred_len] = {
            'ridge': cal_metrics(individual_preds['ridge'].reshape(ori_shape), test_y_reshaped),
            'xgb': cal_metrics(individual_preds['xgb'].reshape(ori_shape), test_y_reshaped), 
            'rf': cal_metrics(individual_preds['rf'].reshape(ori_shape), test_y_reshaped)
        }
        
        # Show results
        baseline_mse = baseline_results[pred_len]['norm']['MSE']
        ensemble_mse = ensemble_results[pred_len]['norm']['MSE']
        improvement = ((baseline_mse - ensemble_mse) / baseline_mse) * 100
        
        print(f"   ✅ Results:")
        print(f"      Baseline MSE: {baseline_mse:.6f}")
        print(f"      Ensemble MSE: {ensemble_mse:.6f}")
        print(f"      Improvement: {improvement:+.2f}%")
    
    return baseline_results, ensemble_results, individual_results

def plot_results(baseline_results, ensemble_results, individual_results):
    """Plot comprehensive results comparison"""
    
    horizons = sorted(ensemble_results.keys())
    
    # Extract MSE values
    baseline_mse = [baseline_results[h]['norm']['MSE'] for h in horizons]
    ensemble_mse = [ensemble_results[h]['norm']['MSE'] for h in horizons]
    ridge_mse = [individual_results[h]['ridge']['MSE'] for h in horizons]
    xgb_mse = [individual_results[h]['xgb']['MSE'] for h in horizons]
    rf_mse = [individual_results[h]['rf']['MSE'] for h in horizons]
    
    # Create plots
    fig, ((ax1, ax2)) = plt.subplots(1, 2, figsize=(15, 5))
    
    # MSE Comparison
    x_pos = np.arange(len(horizons))
    width = 0.15
    
    ax1.bar(x_pos - 2*width, baseline_mse, width, label='TS2Vec+Ridge', alpha=0.8, color='orange')
    ax1.bar(x_pos - width, ridge_mse, width, label='Ridge Only', alpha=0.8, color='lightblue')
    ax1.bar(x_pos, xgb_mse, width, label='XGBoost Only', alpha=0.8, color='lightcoral')
    ax1.bar(x_pos + width, rf_mse, width, label='RF Only', alpha=0.8, color='lightgreen')
    ax1.bar(x_pos + 2*width, ensemble_mse, width, label='Ensemble', alpha=0.8, color='darkgreen')
    
    ax1.set_xlabel('Prediction Horizon')
    ax1.set_ylabel('MSE')
    ax1.set_title('MSE Comparison Across Methods', fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([f'H={h}' for h in horizons])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Improvement percentages
    improvements = [((baseline_results[h]['norm']['MSE'] - ensemble_results[h]['norm']['MSE']) / 
                    baseline_results[h]['norm']['MSE']) * 100 for h in horizons]
    
    colors = ['green' if imp > 0 else 'red' for imp in improvements]
    ax2.bar(x_pos, improvements, color=colors, alpha=0.7)
    ax2.set_xlabel('Prediction Horizon')
    ax2.set_ylabel('Improvement (%)')
    ax2.set_title('Ensemble vs Baseline Improvement', fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f'H={h}' for h in horizons])
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    avg_improvement = np.mean(improvements)
    print(f"\n{'='*80}")
    print(f"🏆 FINAL RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"📈 Average MSE improvement: {avg_improvement:+.2f}%")
    print(f"🥇 Best improvement: {max(improvements):+.2f}% (H={horizons[np.argmax(improvements)]})")
    print(f"🎯 Ensemble weights used: {ENSEMBLE_WEIGHTS}")
    print(f"🚀 Quick mode: {QUICK_MODE}")
    
    if avg_improvement > 0:
        print(f"✅ SUCCESS: Ensemble outperforms TS2Vec baseline!")
    else:
        print(f"⚠️ Mixed results - try tuning ensemble weights or adding more models")

# Run the complete workflow
print("🚀 Starting complete TS2Vec + Ensemble workflow...")

results = run_ts2vec_ensemble_forecasting()

if results is not None:
    print("\n? Evaluating forecasting performance...")
    baseline_results, ensemble_results, individual_results = evaluate_forecasting(results)
    
    print("\n? Plotting results...")
    plot_results(baseline_results, ensemble_results, individual_results)
    
    print(f"\n🎉 WORKFLOW COMPLETED!")
    print(f"💡 To improve results further:")
    print(f"   - Set QUICK_MODE=False for longer training")
    print(f"   - Tune ensemble weights")
    print(f"   - Add more models to ensemble")
    print(f"   - Increase TS2Vec representation dimensions")
else:
    print("❌ Workflow failed - check data loading and setup")